# SVD & Low-Rank Approximation

Companion notebook for the [SVD & Low-Rank Approximation](https://ml-viz-ruby.vercel.app/courses/linear-algebra/04-svd-and-low-rank) lesson on ML Viz.

We'll verify the rotate–stretch–rotate picture, check Eckart–Young numerically, and compress an image with truncated SVD.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("dark_background")
plt.rcParams["figure.facecolor"] = "#0f1117"
plt.rcParams["axes.facecolor"] = "#1a1d27"
plt.rcParams["axes.edgecolor"] = "#2e3347"
plt.rcParams["grid.color"] = "#2e3347"

rng = np.random.default_rng(42)

## Intuition — every matrix is rotate · stretch · rotate

The Singular Value Decomposition says something remarkable: **any** matrix `A`
(square or not) can be written as `A = U Σ Vᵀ`, a **rotation** (`Vᵀ`), an
axis-aligned **stretch** (`Σ`, the non-negative *singular values*), and another
**rotation** (`U`). The singular values rank how much each direction matters, so
keeping only the top few gives the **best possible low-rank approximation** of `A`.
That one idea powers image/data compression, denoising, PCA, latent-factor
recommenders, and the low-rank tricks (like LoRA) used to fine-tune large models.

## 1. SVD from scratch — via the eigendecomposition of AᵀA

The SVD isn't magic; it falls out of eigendecomposition. `AᵀA` is symmetric and
positive-semidefinite, so `eigh` gives it real eigenvalues and orthogonal
eigenvectors. Those eigenvectors are the **right singular vectors** `V`; the
**singular values** are `σ = √eigenvalue` (sorted large→small); and each **left
singular vector** is `uᵢ = A vᵢ / σᵢ`. That's the whole recipe.

In [ ]:
A = rng.normal(size=(5, 3))          # any shape works; here tall (5×3)

def svd_from_scratch(A):
    AtA = A.T @ A                              # (n, n) symmetric PSD
    evals, V = np.linalg.eigh(AtA)             # eigenvectors = right singular vectors
    order = np.argsort(evals)[::-1]            # largest singular value first
    evals, V = evals[order], V[:, order]
    sigma = np.sqrt(np.clip(evals, 0, None))   # singular values = √eigenvalues
    U = np.zeros((A.shape[0], len(sigma)))
    for i, s in enumerate(sigma):
        if s > 1e-12:
            U[:, i] = (A @ V[:, i]) / s         # left singular vector uᵢ = A vᵢ / σᵢ
    return U, sigma, V.T

U, s, Vt = svd_from_scratch(A)
print('singular values          :', s.round(3))
print('U has orthonormal columns:', np.allclose(U.T @ U, np.eye(len(s))))
print('reconstruction U Σ Vᵀ = A:', np.allclose(U @ np.diag(s) @ Vt, A))

**What to notice:** the singular values come out sorted and non-negative, `U`'s columns
are orthonormal, and `U Σ Vᵀ` reassembles `A` exactly — all from one call to `eigh` plus
the recipe above. (In real code you'd never do this: forming `AᵀA` squares the condition
number and loses precision. That's the next section's point.)

## 2. The library way — `np.linalg.svd`, cross-checked

Production code calls `np.linalg.svd`, which uses a numerically stable algorithm that
never forms `AᵀA`. `full_matrices=False` returns the **economy** SVD (just the useful
`k = min(m, n)` singular triplets). The cell asserts our from-scratch singular values
and reconstruction match NumPy's, and confirms `U` and `V` are orthogonal.

In [ ]:
U2, s2, Vt2 = np.linalg.svd(A, full_matrices=False)

assert np.allclose(s, s2), "singular values must match the from-scratch version"
assert np.allclose(U @ np.diag(s) @ Vt, U2 @ np.diag(s2) @ Vt2), "same reconstruction"
print('from-scratch SVD matches np.linalg.svd ✓')

print('U orthogonal :', np.allclose(U2.T @ U2, np.eye(3)))
print('V orthogonal :', np.allclose(Vt2 @ Vt2.T, np.eye(3)))
print('singular values (sorted, non-negative):', s2.round(3))

**What to notice:** identical singular values and reconstruction — our recipe was
correct — and `U`, `V` are orthogonal (rotations). The takeaway: understand the SVD via
`AᵀA`, but *call* `np.linalg.svd`, which is both faster and more accurate.

## 3. Application — the PCA connection

PCA and SVD are the same computation. The **right singular vectors of centered data**
are the principal components, and `σ²/n` are the variances they explain. The cell
computes PCA both ways — covariance eigendecomposition vs. SVD of the centered data —
and prints the variances side by side.

In [ ]:
X = rng.normal(size=(500, 2)) @ np.array([[2.0, 1.2], [0.0, 0.6]])
Xc = X - X.mean(axis=0)

# PCA via covariance eigendecomposition
evals, evecs = np.linalg.eigh(Xc.T @ Xc / len(Xc))
# PCA via SVD
U, s, Vt = np.linalg.svd(Xc, full_matrices=False)

print("eigen variances :", np.sort(evals)[::-1].round(4))
print("SVD variances   :", (s**2 / len(Xc)).round(4))

**What to notice:** the two variance vectors match to four decimals. Running PCA via the
SVD of the data matrix (rather than eigendecomposing the covariance) is the standard,
more numerically stable route — the same reason `np.linalg.svd` beat our `AᵀA` recipe.

## Eckart–Young — truncation is provably optimal

Keep only the top `k` singular triplets and you get `A_k`, the **best** rank-`k`
approximation of `A` in the Frobenius (and spectral) norm. Its error has a clean closed
form: `‖A − A_k‖_F = √(Σ_{i>k} σᵢ²)`, the root-sum-square of the *discarded* singular
values. The cell verifies that formula and then tries — and fails — to beat `A_k` with
20 random rank-`k` matrices.

In [ ]:
A = rng.normal(size=(60, 40)) @ rng.normal(size=(40, 40))
U, s, Vt = np.linalg.svd(A, full_matrices=False)

k = 10
A_k = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
err = np.linalg.norm(A - A_k, "fro")
tail = np.sqrt((s[k:] ** 2).sum())
print(f"truncation error = {err:.4f}, discarded tail = {tail:.4f}")

# try to beat it with random rank-k matrices (spoiler: you can't)
best_random = min(
    np.linalg.norm(A - (rng.normal(size=(60, k)) @ np.linalg.lstsq(rng.normal(size=(60, k)), A, rcond=None)[0]), "fro")
    for _ in range(20)
)
print(f"best of 20 random rank-{k} attempts = {best_random:.4f}  (worse)")

**What to notice:** the measured truncation error equals the discarded-tail RSS exactly,
and the best of 20 random rank-`k` attempts is strictly worse. That's Eckart–Young: no
rank-`k` matrix approximates `A` better than its top-`k` SVD truncation.

## Image compression — watch structure return

A grayscale image is just a matrix, so truncating its SVD compresses it. The cell
rebuilds a structured synthetic image at rank 1, 3, 10, and full rank, labels each with
the fraction of "energy" (`Σσ²`) retained, and plots the singular-value spectrum on a
log scale.

In [ ]:
# build a structured image: gradients + blocks + a bit of noise
n = 128
yy, xx = np.mgrid[0:n, 0:n] / n
img = np.sin(6 * xx) + 0.8 * np.cos(4 * yy) + (xx > 0.5) * (yy > 0.5) * 1.5
img += 0.05 * rng.normal(size=img.shape)

U, s, Vt = np.linalg.svd(img, full_matrices=False)

fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
for ax, k in zip(axes, [1, 3, 10, 128]):
    approx = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
    kept = 100 * (s[:k]**2).sum() / (s**2).sum()
    ax.imshow(approx, cmap="magma")
    ax.set_title(f"rank {k} · {kept:.1f}% energy", fontsize=10)
    ax.axis("off")
plt.suptitle("Truncated SVD: the best rank-k views of the image")
plt.show()

plt.figure(figsize=(7, 3))
plt.semilogy(s, color="#6366f1", lw=2)
plt.xlabel("index")
plt.ylabel("singular value (log)")
plt.title("Spectrum: a few large values, then a noise floor")
plt.grid(alpha=0.4)
plt.show()

**What to notice:** structure returns layer by layer — a handful of components already
capture the gradients and the block, and by rank 10 the image is nearly perfect. The
spectrum explains why: a few large singular values, then a sharp drop to a **noise
floor**. That gap between signal and noise is exactly what makes low-rank compression
(and denoising — dropping the floor) work.

## 4. Tradeoffs & gotchas

- **Shapes: full vs economy.** `np.linalg.svd(A)` returns `U` `(m×m)` and `Vᵀ` `(n×n)`;
  `full_matrices=False` returns the economy `(m×k)` / `(k×n)`. For low-rank work you
  almost always want the economy form (and `svds`/randomized SVD for huge matrices).
- **Rank is a judgement call.** Numerically, "rank" = how many singular values exceed a
  **tolerance**; tiny σ's are usually noise, not signal.
- **Cost.** A full SVD is `O(mn·min(m,n))` — expensive for large dense matrices, which
  is why truncated/randomized SVD exists.
- **Sign ambiguity.** Each `(uᵢ, vᵢ)` pair can flip sign together without changing `A`,
  so singular vectors aren't unique — don't compare them sign-for-sign across libraries.

In [ ]:
A = rng.normal(size=(5, 3))          # a fresh tall matrix, self-contained

# full vs economy shapes
Uf, sf, Vtf = np.linalg.svd(A)
Ue, se, Vte = np.linalg.svd(A, full_matrices=False)
print('full    U,Vᵀ:', Uf.shape, Vtf.shape)      # (5,5) (3,3)
print('economy U,Vᵀ:', Ue.shape, Vte.shape)      # (5,3) (3,3)

# numerical rank from a tolerance: a rank-1 matrix + tiny noise
Alr = np.outer([1., 2., 3.], [4., 5., 6.]) + 1e-12 * rng.normal(size=(3, 3))
sv = np.linalg.svd(Alr, compute_uv=False)
print('singular values:', sv.round(4), '-> numerical rank', int(np.sum(sv > 1e-8)))

# sign ambiguity: flip column 0 of U and row 0 of Vᵀ -> same A
Uc, sc, Vc = np.linalg.svd(A, full_matrices=False)
Uc[:, 0] *= -1; Vc[0, :] *= -1
print('sign-flipped factors still reconstruct A:', np.allclose(Uc @ np.diag(sc) @ Vc, A))

**What to notice:** the full SVD pads `U` and `Vᵀ` out to square (`(5,5)` and
`(3,3)` — the extra `U` columns span the null space); the economy form drops them,
giving `(5,3)` and `(3,3)`. The noisy rank-1 matrix has one big singular value and two
~`1e-12` ones → numerical rank 1. And flipping a matched sign pair still reconstructs
`A` — the decomposition is unique only up to those signs.

## Key takeaways

- **`A = U Σ Vᵀ`** for any matrix: rotate, stretch by the singular values, rotate.
- Singular values are `√`eigenvalues of `AᵀA`, sorted large→small; keeping the top `k`
  gives the **best rank-`k` approximation** (Eckart–Young), with error `√Σ_{i>k}σᵢ²`.
- **PCA is the SVD** of centered data — the numerically preferred route.
- Compression/denoising work because real matrices have a few large singular values and
  a noise floor.
- Use `np.linalg.svd` (not an `AᵀA` recipe); pick economy shapes, judge rank by a σ
  tolerance, and remember singular vectors carry a sign ambiguity.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — Best rank-k approximation

The Eckart–Young theorem says the best rank-$k$ approximation of $A$ keeps the top $k$ singular triplets:

$$A_k = U_{:, :k} \, \Sigma_{:k, :k} \, V^\top_{:k, :}
\qquad\text{with}\qquad
\lVert A - A_k \rVert_F = \sqrt{\textstyle\sum_{i > k} \sigma_i^2}$$

Implement the truncation; the checks verify both the rank **and** that exact error formula.

In [ ]:
def rank_k_approx(A, k):
    """Best rank-k approximation of A (Eckart-Young), built from its SVD."""
    U, s, Vt = np.linalg.svd(np.asarray(A, dtype=float), full_matrices=False)

    # TODO(you): keep the first k columns of U, first k singular values,
    # first k rows of Vt, and multiply them back together
    # (hint: U[:, :k], np.diag(s[:k]), Vt[:k, :])
    return ...

In [ ]:
# Checks — run me
rng = np.random.default_rng(0)
A = rng.standard_normal((8, 6))

A2 = rank_k_approx(A, 2)
assert np.linalg.matrix_rank(A2) == 2, "the result must have rank k"

s = np.linalg.svd(A, compute_uv=False)
expected_err = np.sqrt(np.sum(s[2:] ** 2))
assert abs(np.linalg.norm(A - A2, 'fro') - expected_err) < 1e-9, \
    "Frobenius error must equal √(sum of discarded σ²) — Eckart-Young"

assert np.allclose(rank_k_approx(A, 6), A), "keeping every singular value reconstructs A exactly"
assert np.allclose(rank_k_approx(A, 0), 0), "rank-0 'approximation' is just the zero matrix (degenerate case)"

single = rank_k_approx(np.array([[3.0]]), 1)
assert np.allclose(single, [[3.0]]), "a 1x1 matrix is its own rank-1 approximation"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def rank_k_approx(A, k):
    U, s, Vt = np.linalg.svd(np.asarray(A, dtype=float), full_matrices=False)
    return U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
```

</details>

### Exercise 2 — Compression ratio

Storing a full $m \times n$ matrix takes $mn$ numbers. The rank-$k$ version stores only the factors: $U_k$ ($m \cdot k$), the singular values ($k$), and $V_k^\top$ ($k \cdot n$). Compute how many times smaller that is — and notice the punchline of the last check: for $k$ too close to full rank, "compression" actually costs **more** than the original.

In [ ]:
def compression_ratio(m, n, k):
    """How many times smaller rank-k SVD storage is vs. the full m x n matrix."""
    # TODO(you): numbers needed for the full matrix
    full = ...

    # TODO(you): numbers needed for the rank-k factors: m*k + k + k*n
    compressed = ...

    return full / compressed

In [ ]:
# Checks — run me
assert abs(compression_ratio(100, 100, 10) - 10000 / 2010) < 1e-12, "100x100 at k=10 -> ~5x smaller"
assert abs(compression_ratio(1000, 500, 20) - 500000 / 30020) < 1e-12, "1000x500 at k=20 -> ~16.7x smaller"
assert compression_ratio(10, 10, 8) < 1, "k too close to full rank costs MORE than storing A"
assert abs(compression_ratio(1, 1, 1) - 1 / 3) < 1e-12, "a 1x1 matrix: 'compressing' still needs 3 numbers -> ratio < 1"
assert compression_ratio(50, 50, 1) > 1, "k=1 on a square matrix is always a big win"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def compression_ratio(m, n, k):
    full = m * n
    compressed = m * k + k + k * n
    return full / compressed
```

</details>

---
## 🎯 Extra practice — SVD variants & sparse formats

Five more routines from [DML-OpenProblem](https://github.com/Open-Deep-ML/DML-OpenProblem)
that round out this lesson: an alternative way to *compute* a 2x2 SVD, the two
statistics SVD/PCA are usually applied to, and the sparse-matrix formats that let
SVD-style factorizations scale to huge, mostly-zero datasets.

- **DML 12** — `singular-value-decomposition-svd` (2x2 via a single Jacobi rotation
  — no `np.linalg.svd` allowed)
- **DML 10** — `calculate-covariance-matrix`
- **DML 37** — `calculate-correlation-matrix`
- **DML 65** — `implement-compressed-row-sparse-matrix-csr-format`
- **DML 67** — `implement-compressed-column-sparse-matrix-format-csc`

In [ ]:
def svd_2x2_jacobi(A):
    """DML 12 — approximate 2x2 SVD using ONE Jacobi rotation (no np.linalg.svd)."""
    A = np.asarray(A, dtype=float)
    AtA = A.T @ A
    # TODO(you): the rotation angle that diagonalizes A^T A in one step
    if abs(AtA[0, 0] - AtA[1, 1]) < 1e-15:
        theta = np.pi / 4
    else:
        theta = 0.5 * np.arctan2(2 * AtA[0, 1], AtA[0, 0] - AtA[1, 1])
    c, s = np.cos(theta), np.sin(theta)
    V = np.array([[c, -s], [s, c]])
    # TODO(you): singular values are the column norms of A @ V
    AV = A @ V
    sigma = np.linalg.norm(AV, axis=0)
    U = AV / sigma
    return U, sigma, V.T


def calculate_covariance_matrix(vectors):
    """DML 10 — covariance matrix for a list of feature vectors (each a list of observations)."""
    X = np.array(vectors, dtype=float)
    # TODO(you): center each feature (row) around its own mean
    Xc = X - X.mean(axis=1, keepdims=True)
    n = X.shape[1]
    return (Xc @ Xc.T) / (n - 1)


def calculate_correlation_matrix(X, Y=None):
    """DML 37 — Pearson correlation matrix between columns of X (and Y, if given)."""
    X = np.asarray(X, dtype=float)
    Y = X if Y is None else np.asarray(Y, dtype=float)
    Xc = X - X.mean(axis=0)
    Yc = Y - Y.mean(axis=0)
    # TODO(you): standardize each column by its own std before taking the dot product
    Xs = Xc / X.std(axis=0)
    Ys = Yc / Y.std(axis=0)
    return (Xs.T @ Ys) / X.shape[0]


def compressed_row_sparse_matrix(dense_matrix):
    """DML 65 — dense matrix -> CSR (values, column indices, row pointer)."""
    values, col_idx, row_ptr = [], [], [0]
    for row in dense_matrix:
        for j, v in enumerate(row):
            # TODO(you): only non-zero entries are stored
            if v != 0:
                values.append(v)
                col_idx.append(j)
        row_ptr.append(len(values))
    return values, col_idx, row_ptr


def compressed_col_sparse_matrix(dense_matrix):
    """DML 67 — dense matrix -> CSC (values, row indices, column pointer)."""
    n_rows = len(dense_matrix)
    n_cols = len(dense_matrix[0]) if n_rows else 0
    values, row_idx, col_ptr = [], [], [0]
    for j in range(n_cols):
        for i in range(n_rows):
            v = dense_matrix[i][j]
            # TODO(you): walk column-major this time, still only storing non-zeros
            if v != 0:
                values.append(v)
                row_idx.append(i)
        col_ptr.append(len(values))
    return values, row_idx, col_ptr

In [ ]:
# Checks — run me
U, S, VT = svd_2x2_jacobi([[2, 1], [1, 2]])
assert np.allclose(U @ np.diag(S) @ VT, [[2, 1], [1, 2]]), "DML 12: reconstruction must equal A"
assert np.allclose(sorted(S), [1.0, 3.0]), "DML 12 example: singular values 3 and 1"
assert np.allclose(U.T @ U, np.eye(2)), "U must stay orthogonal"

cov = calculate_covariance_matrix([[1, 2, 3], [4, 5, 6]])
assert np.allclose(cov, [[1.0, 1.0], [1.0, 1.0]]), "DML 10 example"
identical = calculate_covariance_matrix([[5, 5, 5], [1, 2, 3]])
assert np.allclose(identical[0], [0, 0]), "a constant feature (zero variance) has zero covariance with anything"

corr = calculate_correlation_matrix(np.array([[1, 2], [3, 4], [5, 6]]))
assert np.allclose(corr, [[1.0, 1.0], [1.0, 1.0]]), "DML 37 example"
anti = calculate_correlation_matrix(np.array([[1.0, 3.0], [2.0, 2.0], [3.0, 1.0]]))
assert abs(anti[0, 1] + 1) < 1e-9, "a perfectly inverse relationship -> correlation -1"

dm = [[1, 0, 0, 0], [0, 2, 0, 0], [3, 0, 4, 0], [1, 0, 0, 5]]
vals, col_idx, row_ptr = compressed_row_sparse_matrix(dm)
assert (vals, col_idx, row_ptr) == ([1, 2, 3, 4, 1, 5], [0, 1, 0, 2, 0, 3], [0, 1, 2, 4, 6]), "DML 65 example"
z_vals, z_col, z_ptr = compressed_row_sparse_matrix([[0, 0], [0, 0]])
assert z_vals == [] and z_ptr == [0, 0, 0], "an all-zero matrix stores nothing but still has a row pointer"

dm2 = [[0, 0, 3, 0], [1, 0, 0, 4], [0, 2, 0, 0]]
vals2, row_idx2, col_ptr2 = compressed_col_sparse_matrix(dm2)
assert (vals2, row_idx2, col_ptr2) == ([1, 2, 3, 4], [1, 2, 0, 1], [0, 1, 2, 3, 4]), "DML 67 example"

print("✅ Extra-practice SVD variants & sparse formats passed")

<details>
<summary>💡 Show solution</summary>

```python
def svd_2x2_jacobi(A):
    A = np.asarray(A, dtype=float)
    AtA = A.T @ A
    if abs(AtA[0, 0] - AtA[1, 1]) < 1e-15:
        theta = np.pi / 4
    else:
        theta = 0.5 * np.arctan2(2 * AtA[0, 1], AtA[0, 0] - AtA[1, 1])
    c, s = np.cos(theta), np.sin(theta)
    V = np.array([[c, -s], [s, c]])
    AV = A @ V
    sigma = np.linalg.norm(AV, axis=0)
    U = AV / sigma
    return U, sigma, V.T


def calculate_covariance_matrix(vectors):
    X = np.array(vectors, dtype=float)
    Xc = X - X.mean(axis=1, keepdims=True)
    n = X.shape[1]
    return (Xc @ Xc.T) / (n - 1)


def calculate_correlation_matrix(X, Y=None):
    X = np.asarray(X, dtype=float)
    Y = X if Y is None else np.asarray(Y, dtype=float)
    Xc = X - X.mean(axis=0)
    Yc = Y - Y.mean(axis=0)
    Xs = Xc / X.std(axis=0)
    Ys = Yc / Y.std(axis=0)
    return (Xs.T @ Ys) / X.shape[0]


def compressed_row_sparse_matrix(dense_matrix):
    values, col_idx, row_ptr = [], [], [0]
    for row in dense_matrix:
        for j, v in enumerate(row):
            if v != 0:
                values.append(v)
                col_idx.append(j)
        row_ptr.append(len(values))
    return values, col_idx, row_ptr


def compressed_col_sparse_matrix(dense_matrix):
    n_rows = len(dense_matrix)
    n_cols = len(dense_matrix[0]) if n_rows else 0
    values, row_idx, col_ptr = [], [], [0]
    for j in range(n_cols):
        for i in range(n_rows):
            v = dense_matrix[i][j]
            if v != 0:
                values.append(v)
                row_idx.append(i)
        col_ptr.append(len(values))
    return values, row_idx, col_ptr
```

</details>

**Next:** [PCA](https://ml-viz-ruby.vercel.app/courses/pca-dimensionality/01-pca) puts this machinery to work on data, and the [course quiz](https://ml-viz-ruby.vercel.app/courses/linear-algebra/05-quiz) checks the whole course.